In [13]:
import os
import torch
import wandb
import evaluate
import numpy as np
from datasets import load_dataset

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)

from peft import LoraConfig
from trl import SFTTrainer

### 경로 설정

In [14]:
base_model  = "deepseek-ai/deepseek-coder-1.3b-instruct" 

train_path= "/Users/jangjunmin/Documents/code/defAPI_project/Create_data/ft_data.jsonl"
test_path = "/Users/jangjunmin/Documents/code/defAPI_project/Create_data/test_data.jsonl"

new_model = "deekseek-coder-1.3b-instruct-finetuned-gbsw"

### Parameter 설정

In [15]:
lora_r = 32 # 값이 클수록 더 많은 수정 일어나 모델이 복잡해짐
lora_alpha = 8 # LoRA가 적용될때 가중치에 얼마나 영향을 미칠지
lora_dropout = 0.06 # 과적합 방지 위해 드롭아웃 비율 설정

In [16]:
use_4bit = True #4bit 양자화 사용 여부
bnb_4bit_compute_dtype = "float16" #dtype 설정
bnb_4bit_quant_type = "nf4" #양자화 유형 설정
use_nested_quant = False #중첩 양자화 사용 여부

In [17]:
output_dir = "./results" #모델 예측결과, 체크포인트 저장 디렉토리

num_train_epochs = 1 # 학습 epoch 수
per_device_train_batch_size = 1 # 훈련용 배치 크기
per_device_eval_batch_size = 1  # 평가용 배치 크기
gradient_accumulation_steps = 2  # 그래디언트를 누적할 업데이트 스텝 횟수
gradient_checkpointing = True  # 그래디언트 체크포인트 활성화
max_grad_norm = 0.6  # 그래디언트 클리핑은 그래디언트의 크기를 제한하여 훈련 중 안정성을 높임.
learning_rate = 2e-6 # 초기 학습률 AdamW 옵티마이저
weight_decay = 0.001 # bias/LayerNorm 가중치를 제외하고 모든 레이어에 적용할 Weight decay 값
optim = "paged_adamw_32bit"  # 옵티마이저 설정
lr_scheduler_type = "cosine"  # 학습률 스케줄러의 유형 설정, 여기서는 코사인 스케줄러 사용
max_steps = -1 # 훈련 스텝 수(num_train_epochs 재정의)
warmup_ratio = 0.03 # (0부터 learning rate까지) 학습 초기에 학습률을 점진적으로 증가시키 linear warmup 스텝의 Ratio
group_by_length = True  # 시퀀스를 동일한 길이의 배치로 그룹화, 메모리 절약 및 훈련 속도를 높임
save_steps = 0  # X 업데이트 단계마다 체크포인트 저장
logging_steps = 500  # 매 X 업데이트 스텝 로그

In [18]:
# 최대 시퀀스 길이 설정
max_seq_length = 1024

# 동일한 입력 시퀀스에 여러 개의 짧은 예제를 넣어 효율성을 높일 수 있음
packing = False  

# GPU 0 전체 모델 로드 
device_map = {"": 0}

In [19]:
os.environ["WANDB_PROJECT"]="defAPI"
os.environ["WANDB_LOG_MODEL"]="true"
os.environ["WANDB_WATCH"]="false"

In [20]:
bnb_config = BitsAndBytesConfig(
  load_in_4bit = use_4bit,
  bnb_4bit_quant_type = bnb_4bit_quant_type,
  bnb_4bit_compute_dtype= getattr(torch, bnb_4bit_compute_dtype),
  bnb_4bit_use_double_quant= use_nested_quant
)

In [21]:
dataset = load_dataset("json",
                       data_files={
                         "train": train_path,
                          "test": test_path})
print("="*100)
print(dataset["train"][0])
print("="*100)
print(dataset["test"][0])

{'prompt': '### Instruction:\n아래 코드를 보안 취약점이 없도록 수정해줘. 변경된 라인에 주석으로 이유를 적어줘.\n\n### Input:\nimport java.sql.*;\n\npublic class SQLInjectionVulnerable {\n    public static void main(String[] args) {\n        String userInput = "admin\'; DROP TABLE users; --"; // User input with malicious SQL injection payload\n\n        try {\n            Connection connection = DriverManager.getConnection("jdbc:mysql://localhost:3306/mydatabase", "username", "password");\n            Statement statement = connection.createStatement();\n\n            // Vulnerable code - concatenating user input directly into the SQL query\n            String query = "SELECT * FROM users WHERE username=\'" + userInput + "\'";\n            ResultSet resultSet = statement.executeQuery(query);\n\n            while (resultSet.next()) {\n                System.out.println("Username: " + resultSet.getString("username"));\n                System.out.println("Password: " + resultSet.getString("password"));\n            }\n\n   

In [1]:
# 데이터셋 정보 확인
print("="*100)
print(f"훈련 데이터셋 크기: {len(dataset['train'])} 샘플")
print("\n훈련 데이터셋 특성:")
print(dataset["train"].features)
print("="*100)
print(f"테스트 데이터셋 크기: {len(dataset['test'])} 샘플")
print("\n테스트 데이터셋 특성:")
print(dataset["test"].features)
print("="*100)

NameError: name 'dataset' is not defined

In [25]:
model = AutoModelForCausalLM.from_pretrained(
  base_model,
  quantization_config=bnb_config,
  device_map="mps"
)

model.config.use_cache = False
model.config.pretraining_tp = 1

ImportError: Using `bitsandbytes` 4-bit quantization requires the latest version of bitsandbytes: `pip install -U bitsandbytes`

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
  base_model,
  trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
def preprocess_fn(example):
    inputs = example["prompt"] + example["completion"]
    tokenized = tokenizer(
        inputs,
        truncation=True,
        max_length=max_seq_length
    )
    
    return tokenized

tokenized_dataset = dataset.map(preprocess_fn, remove_columns=dataset["train"].column_names)

In [ ]:
train_data = tokenized_dataset["train"].shuffle(seed=42)
test_data = tokenized_dataset["test"].shuffle(seed=42)

In [ ]:
peft_params = LoraConfig(
  lora_alpha=lora_alpha,
  lora_dropout=lora_dropout,
  r=lora_r,
  bias="none",
  task_type="CAUSAL_LM"
) 

In [ ]:
training_params = TrainingArguments(
    output_dir=output_dir,
    report_to="wandb",
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    fp16=False,
    bf16=False,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=group_by_length,
    lr_scheduler_type=lr_scheduler_type,
)

In [ ]:
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_eval(eval_pred):
  logits, labels = eval_pred

  predictions = np.argmax(logits, axis=-1)
  
  bleu_score = bleu.compute(predictions=predictions, references=labels)
  rouge_score = rouge.compute(predictions=predictions, references=labels)
  accuracy_score = accuracy.compute(predictions=predictions, references=labels)
  f1_score = f1.compute(predictions=predictions, references=labels)

  return bleu_score, rouge_score, accuracy_score, f1_score

### 학습

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=test_data,
    peft_config=peft_params,
    args=training_params,
    compute_eval=compute_eval
)

In [ ]:
trainer.train()

trainer.model.save_pretrained(new_model)

wandb.finish()

### LoRA 가중치 모델이랑 합치기

In [ ]:
# from peft import PeftModel

# model = AutoModelForCausalLM.from_pretrained(
#     base_model,
#     low_cpu_mem_usage=True,
#     return_dict=True,
#     dtype=torch.float16
# )

# model = PeftModel.from_pretrained(model, new_model)

In [ ]:
# model = model.merge_and_unload()